In [77]:
import pandas as pd
import numpy as np

from sklearn.base import BaseEstimator, TransformerMixin
from catboost import CatBoostClassifier
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split

In [93]:
cat_features = ['Sex',
 'Chest pain type',
 'FBS over 120',
 'EKG results',
 'Exercise angina',
 'Slope of ST']

In [94]:
class FeatureEngineer(BaseEstimator, TransformerMixin):
    def __init__(self):
        pass

    def fit(self, X, y=None):
        return self
    def transform(self, X):

        X = X.copy()

        if 'id' in X.columns:
            X = X.drop('id', axis=1)

        # Defina abaixo as manipulações de teste nos dados:

        X['Sex'] = list(map(lambda x: 'M' if x==1 else 'F', X['Sex']))
        #Transforming into category, thus 0 and 1 doesn't have distance interpretation on sex
        
        X['Chest pain type'] = list(map(lambda x: 'Light' if x==1 else 'Common' if x==2 else 'Hard' if x==3 else 'Dangerous', X['Chest pain type']))
        #Transforming into category, same reason
        
        X['FBS over 120'] = list(map(lambda x: 'Y' if x==1 else 'N', X['FBS over 120']))
        #Transforming into category, same reason
        
        X['Exercise angina'] = list(map(lambda x: 'Y' if x==1 else 'N', X['Exercise angina']))
        # Transforming into category, same reason
        
        X['Slope of ST'] = list(map(lambda x: 'Upsloping' if x==1 else 'Flat' if x==2 else 'Downsloping', X['Slope of ST']))
        # Transforming into category, same reason
        
        X['EKG results'] = list(map(lambda x: 'N' if x in [0,1] else 'P', X['EKG results']))
        # Merging 0 and 1 to 'N' thus 0 and 1 have the same relation with Heart Disease and, transforming into category
        # thus, here also, there's no distance relation between 0, 1 and 2

        # Define bellow transformations that are not validated yet. therefor, test transformations.

        X['log_HR'] = np.log1p(X['Max HR'])

        try:
            X['Heart Disease'] = X['Heart Disease'].map({'Presence': 1, 'Absence': 0})
        except Exception as e:
            print(f'O df não possui a coluna Heart Disease, portanto essa etapa foi ignorada (erro {e}) ')
        return X

def xy_extract(df):
    target = 'Heart Disease'
    return df.drop(target, axis=1), df[target]


def traintest_extract(X, y):
    X_train, X_test, y_train, y_test = train_test_split(X, y, train_size=0.8)
    return X_train, X_test, y_train, y_test
    

In [95]:
df = pd.read_csv('train.csv')
params = {'iterations': 1233, 'depth': 5, 'learning_rate': 0.05563228358640898, 'l2_leaf_reg': 1.2388924496891942, 'border_count': 157, 'verbose': 200, 'cat_features': cat_features }


In [96]:


pipe = Pipeline([
    ('fe', FeatureEngineer()),
    ('model', CatBoostClassifier(**params))
])

df = df.drop('id', axis=1)

X, y = xy_extract(df)

display(X)
X_train, X_test, y_train, y_test = traintest_extract(X, y)



,Age,Sex,Chest pain type,BP,Cholesterol,FBS over 120,EKG results,Max HR,Exercise angina,ST depression,Slope of ST,Number of vessels fluro,Thallium
0,58,1,4,152,239,0,0,158,1,3.6,2,2,7
1,52,1,1,125,325,0,2,171,0,0.0,1,0,3
2,56,0,2,160,188,0,2,151,0,0.0,1,0,3
3,44,0,3,134,229,0,2,150,0,1.0,2,0,3
4,58,1,4,140,234,0,2,125,1,3.8,2,3,3
...,...,...,...,...,...,...,...,...,...,...,...,...,...
629995,56,0,1,110,226,0,0,132,0,0.0,1,0,7
629996,54,1,4,128,249,1,2,150,0,0.0,2,0,3
629997,67,1,4,130,275,0,0,149,0,0.0,1,2,7
629998,52,1,4,140,199,0,2,157,0,0.0,1,0,6


In [97]:
cat_features

['Sex',
 'Chest pain type',
 'FBS over 120',
 'EKG results',
 'Exercise angina',
 'Slope of ST']

In [98]:
test = pd.read_csv('test.csv')

test_transformed = pipe.named_steps['fe'].transform(test)

test_transformed.info()

O df não possui a coluna Heart Disease, portanto essa etapa foi ignorada (erro 'Heart Disease') 
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 270000 entries, 0 to 269999
Data columns (total 14 columns):
 #   Column                   Non-Null Count   Dtype  
---  ------                   --------------   -----  
 0   Age                      270000 non-null  int64  
 1   Sex                      270000 non-null  object 
 2   Chest pain type          270000 non-null  object 
 3   BP                       270000 non-null  int64  
 4   Cholesterol              270000 non-null  int64  
 5   FBS over 120             270000 non-null  object 
 6   EKG results              270000 non-null  object 
 7   Max HR                   270000 non-null  int64  
 8   Exercise angina          270000 non-null  object 
 9   ST depression            270000 non-null  float64
 10  Slope of ST              270000 non-null  object 
 11  Number of vessels fluro  270000 non-null  int64  
 12  Thallium         

In [103]:
X_test_transformed = pipe.named_steps['fe'].transform(X_test)

model = pipe.fit(
    X_train,
    y_train,
    model__eval_set=(X_test_transformed, y_test),
    model__cat_features=cat_features
)

O df não possui a coluna Heart Disease, portanto essa etapa foi ignorada (erro 'Heart Disease') 
O df não possui a coluna Heart Disease, portanto essa etapa foi ignorada (erro 'Heart Disease') 
0:	learn: 0.6384503	test: 0.6386232	best: 0.6386232 (0)	total: 285ms	remaining: 5m 50s
200:	learn: 0.2698234	test: 0.2710374	best: 0.2710374 (200)	total: 37s	remaining: 3m 10s
400:	learn: 0.2675668	test: 0.2691823	best: 0.2691823 (400)	total: 1m 13s	remaining: 2m 33s
600:	learn: 0.2663935	test: 0.2684386	best: 0.2684386 (600)	total: 1m 52s	remaining: 1m 58s
800:	learn: 0.2656438	test: 0.2681550	best: 0.2681550 (800)	total: 2m 30s	remaining: 1m 21s
1000:	learn: 0.2649847	test: 0.2679843	best: 0.2679833 (998)	total: 3m 10s	remaining: 44.3s
1200:	learn: 0.2644270	test: 0.2679187	best: 0.2679148 (1195)	total: 3m 50s	remaining: 6.13s
1232:	learn: 0.2643455	test: 0.2679009	best: 0.2679003 (1227)	total: 3m 56s	remaining: 0us

bestTest = 0.2679002778
bestIteration = 1227

Shrink model to first 1228 iter

In [104]:
best_iter = pipe.named_steps['model'].get_feature_importance()
best_iter

array([ 3.81704664,  5.73641299, 16.70271786,  0.87181666,  1.9573489 ,
        0.04990815,  1.19409042,  9.05046282,  6.8991254 ,  6.74664028,
        5.86787808, 10.28214108, 19.21817772, 11.606233  ])

In [100]:
X_train.info()

<class 'pandas.core.frame.DataFrame'>
Index: 504000 entries, 615990 to 92331
Data columns (total 13 columns):
 #   Column                   Non-Null Count   Dtype  
---  ------                   --------------   -----  
 0   Age                      504000 non-null  int64  
 1   Sex                      504000 non-null  int64  
 2   Chest pain type          504000 non-null  int64  
 3   BP                       504000 non-null  int64  
 4   Cholesterol              504000 non-null  int64  
 5   FBS over 120             504000 non-null  int64  
 6   EKG results              504000 non-null  int64  
 7   Max HR                   504000 non-null  int64  
 8   Exercise angina          504000 non-null  int64  
 9   ST depression            504000 non-null  float64
 10  Slope of ST              504000 non-null  int64  
 11  Number of vessels fluro  504000 non-null  int64  
 12  Thallium                 504000 non-null  int64  
dtypes: float64(1), int64(12)
memory usage: 53.8 MB


In [102]:
X_test.info()

<class 'pandas.core.frame.DataFrame'>
Index: 126000 entries, 485445 to 517726
Data columns (total 13 columns):
 #   Column                   Non-Null Count   Dtype  
---  ------                   --------------   -----  
 0   Age                      126000 non-null  int64  
 1   Sex                      126000 non-null  int64  
 2   Chest pain type          126000 non-null  int64  
 3   BP                       126000 non-null  int64  
 4   Cholesterol              126000 non-null  int64  
 5   FBS over 120             126000 non-null  int64  
 6   EKG results              126000 non-null  int64  
 7   Max HR                   126000 non-null  int64  
 8   Exercise angina          126000 non-null  int64  
 9   ST depression            126000 non-null  float64
 10  Slope of ST              126000 non-null  int64  
 11  Number of vessels fluro  126000 non-null  int64  
 12  Thallium                 126000 non-null  int64  
dtypes: float64(1), int64(12)
memory usage: 13.5 MB


In [ ]:
X_train.head(1)

In [ ]:
X_test.head()